# SSW60 Test-Set Spectrogram Generation

This notebook prepares the Sapsucker Woods 60 (SSW60) audio test split as
Mel-spectrograms for evaluation of our iNatSounds trained models. It uses 
the iNatSounds preprocessing pipeline `generate_spectrograms.py`
script with the same parameters.

We only look at the test split i.e. 1264 out of the 3861 SSW60 audio files.

We used Google Colab Free to create this spectrogram dataset.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Download the SSW60 archive

The full dataset is 31 GB and downloads in c.25 minutes on Colab's connection.

In [ ]:
!mkdir -p /content/drive/MyDrive/iNatSounds/ssw60
!wget --show-progress -O /content/drive/MyDrive/iNatSounds/ssw60/ssw60.tar.gz \
    https://ml-inat-competition-datasets.s3.amazonaws.com/ssw60/ssw60.tar.gz


## 3. Selectively extract the audio components

We only need the audio data for our pipeline: `audio_ml/`, `audio_ml.csv`,
`taxa.csv`, thus, we do not use the video and image components (which take up the 
majority of the space).


In [ ]:
!tar -xzf /content/drive/MyDrive/iNatSounds/ssw60/ssw60.tar.gz \
    -C /content/drive/MyDrive/iNatSounds \
    ssw60/audio_ml/ ssw60/audio_ml.csv ssw60/taxa.csv


## 4. Sanity checks

We check that the extraction results in everything we expect: the SSW60 README claims
3861 audio files, with the train/test split encoded in the `split` column of
`audio_ml.csv`.


In [ ]:
!find /content/drive/MyDrive/iNatSounds/ssw60/audio_ml -name "*.wav" | wc -l


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/iNatSounds/ssw60/audio_ml.csv")
print(df["split"].value_counts())


## 5. Filter to the test split

The SSW60 archive has all 3861 audio files in a single folder `audio_ml/`. 
We filter `audio_ml.csv` to `split == "test"` and copy those
1264 WAVs audio files into a separate directory `ssw60_audio_test_real/`. 


In [ ]:
!pip install librosa==0.10.2 -q


In [ ]:
import os
import shutil
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/iNatSounds/ssw60/audio_ml.csv")
test_ids = df[df["split"] == "test"]["asset_id"].tolist()

out_dir = "/content/drive/MyDrive/iNatSounds/ssw60/ssw60_audio_test_real"
os.makedirs(out_dir, exist_ok=True)

for i, asset_id in enumerate(test_ids):
    src = f"/content/drive/MyDrive/iNatSounds/ssw60/audio_ml/{asset_id}.wav"
    dst = f"{out_dir}/{asset_id}.wav"
    shutil.copy2(src, dst)
    if (i + 1) % 100 == 0:
        print(f"{i + 1}/{len(test_ids)} copied")

print("Done.")


## 6. Generate spectrograms

We now use the same `generate_spectrograms.py` we use for iNatSounds to generate
the spectrograms.

In [ ]:
!python /content/drive/MyDrive/iNatSounds/ssw60/generate_spectrograms.py \
    --input_dir  /content/drive/MyDrive/iNatSounds/ssw60/ssw60_audio_test_real \
    --output_dir /content/drive/MyDrive/iNatSounds/ssw60/ssw60_spectrograms \
    --num_workers 4
